In [1]:
from agents.text import TextAgent, TextModelConfig

In [ ]:
vllm_config = TextModelConfig(
    backend="vllm",
    model_name="google/gemma-3-1b-it",
    model_path="google/gemma-3-1b-it",
    device="mps",
    torch_dtype="float16",
    max_new_tokens=256,
    temperature=0.7,
    llamacpp_params={
        "n_gpu_layers": 1,
        "tensor_parallel_size": 1,  # Явное указание для MPS
        "mps": True
    },
    disable_distributed=True,
    quantized=False  # Отключаем квантование для MPS
)

agent = TextAgent(
    config=vllm_config,
    title_params={
        "temperature": 0.3,
        "max_new_tokens": 30,
        "top_p": 0.9,
        "repetition_penalty": 1.2,
        "stop_sequences": ["\n"]
    }
)

INFO 03-13 23:40:01 __init__.py:207] Automatically detected platform cpu.
INFO 03-13 23:40:02 config.py:2417] For macOS with Apple Silicon, currently bfloat16 is not supported. Setting dtype to float16.
WARNING 03-13 23:40:02 config.py:2448] Casting torch.bfloat16 to torch.float16.
INFO 03-13 23:40:06 config.py:549] This model supports multiple tasks: {'embed', 'classify', 'generate', 'score', 'reward'}. Defaulting to 'generate'.
WARNING 03-13 23:40:06 config.py:685] Async output processing is not supported on the current platform type cpu.
WARNING 03-13 23:40:06 cpu.py:63] CUDA graph is not supported on CPU, fallback to the eager mode.
WARNING 03-13 23:40:06 cpu.py:78] Environment variable VLLM_CPU_KVCACHE_SPACE (GB) for CPU backend is not set, using 4 by default.
WARNING 03-13 23:40:06 cpu.py:99] uni is not supported on CPU, fallback to mp distributed executor backend.
INFO 03-13 23:40:06 importing.py:16] Triton not installed or not compatible; certain GPU-related functions will not 

[W313 23:41:08.187998000 TCPStore.cpp:141] [c10d] recvValue failed on SocketImpl(fd=89, addr=[::240.0.0.2]:58024, remote=[::ffff:240.0.0.2]:57899): Connection reset by peer
Exception raised from recvBytes at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/distributed/c10d/Utils.hpp:668 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__1::basic_string<char, std::__1::char_traits<char>, std::__1::allocator<char>>) + 52 (0x1088569ec in libc10.dylib)
frame #1: void c10d::tcputil::recvBytes<unsigned int>(int, unsigned int*, unsigned long) + 484 (0x1474a9a44 in libtorch_cpu.dylib)
frame #2: unsigned int c10d::detail::TCPClient::receiveValue<unsigned int>() + 40 (0x1474a9704 in libtorch_cpu.dylib)
frame #3: c10d::TCPStore::ping() + 196 (0x1474a82c8 in libtorch_cpu.dylib)
frame #4: c10d::TCPStore::TCPStore(std::__1::basic_string<char, std::__1::char_traits<char>, std::__1::allocator<char>>, c10d::TCPStoreOptions const&) + 1232 (0x1474a7158 in libtorch_cpu.

In [ ]:
# Генерация заголовка
title = agent.generate_title("Future of AI in healthcare")
print(f"Generated Title: {title}")
# Output: "AI-Driven Innovations Transforming Healthcare Delivery"

# Генерация контента
content = agent.generate_content(
    "Technical explanation of neural networks",
    format_hint="bullet_list"
)
print("Generated Content:")
print(content)
"""
- Neural networks are computational models inspired by biological neurons
- Consist of interconnected layers (input, hidden, output)
- Use activation functions like ReLU and Sigmoid for non-linear transformations
- Trained via backpropagation and gradient descent optimization
"""

In [ ]:
# Конфигурация для Yandex GPT API
import requests


yandex_config = TextModelConfig(
    backend="api",
    api_base="https://llm.api.cloud.yandex.net/llm/v1alpha/instruct",
    api_key="your_iam_token",
    model_name="yandexgpt-lite"  # Бесплатная версия
)

# Инициализация агента
agent = TextAgent(yandex_config)

# Пример генерации
def generate_yandex_gpt(prompt: str) -> str:
    headers = {
        "Authorization": f"Bearer {agent.config.api_key}",
        "Content-Type": "application/json"
    }
    
    data = {
        "model": agent.config.model_name,
        "messages": [{
            "role": "user",
            "content": prompt
        }],
        "temperature": 0.7,
        "max_tokens": 300
    }
    
    response = requests.post(
        agent.config.api_base,
        headers=headers,
        json=data
    )
    return response.json()["result"]["alternatives"][0]["message"]["text"]

# Использование
title = generate_yandex_gpt("Generate title about AI in healthcare")
content = generate_yandex_gpt("Explain neural networks for beginners")